# Rolling LightGBM Prototype for Storage Protection

This notebook prototypes a more conservative planning workflow while preserving the existing artifact pipeline as the benchmark.

## Prototype Enhancements

- Weekly LightGBM refits on expanding history
- Short-horizon recursive forecasts
- Conservative demand uplift above the point forecast
- Fallback demand assumption beyond the short modeled horizon
- Daily receding-horizon optimization
- Positive reserve floor with penalty-backed slack

## Objective

The goal is not only to reduce tariff violations, but also to reduce the risk of storage-bank depletion that appeared in the original optimized backtest. The comparison below focuses on whether violations occurred, not on estimated cash-out dollars.

In [9]:
from pathlib import Path
import sys

import altair as alt
import pandas as pd
import polars as pl
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "silver" / "forecast_artifacts").exists():
            return candidate
    msg = "Could not locate repository root."
    raise FileNotFoundError(msg)


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

alt.data_transformers.disable_max_rows()

from dgup._internal.forecasting import _build_penalty_flag_report, _run_rolling_lightgbm_planning_prototype
from dgup._internal.storage import _reconstruct_storage

In [10]:
ARTIFACT_DIR = REPO_ROOT / "data" / "silver" / "forecast_artifacts"
USAGE_PATH = REPO_ROOT / "data" / "silver" / "uta_gas_usage.parquet"

baseline_delivery_plan = pl.read_parquet(ARTIFACT_DIR / "forecast-v2-delivery-plan.parquet").to_pandas()
baseline_delivery_summary = pl.read_parquet(ARTIFACT_DIR / "forecast-v2-delivery-summary.parquet").to_pandas()
usage_frame = (
    pl.read_parquet(USAGE_PATH)
    .with_columns((pl.col("Usage - 1") + pl.col("Usage - 2") + pl.col("Usage - 2_1")).alias("Total Usage"))
    .to_pandas()
)

for frame in (baseline_delivery_plan, baseline_delivery_summary, usage_frame):
    if "Date" in frame.columns:
        frame["Date"] = pd.to_datetime(frame["Date"])

prototype = _run_rolling_lightgbm_planning_prototype(
    start_date="2024-01-01",
    end_date="2024-11-30",
    retrain_frequency_days=7,
    short_horizon_days=7,
    reserve_floor=2500.0,
)

prototype.delivery_summary

,strategy,mean_delivery,mean_ending_balance,injection_limit_exceeded,withdrawal_limit_exceeded,below_min_inventory,above_max_inventory,balance_below_zero,balance_above_capacity
0,actual_delivery,1870.955337,61606.816509,88,48,0,1,0,0
1,rolling_lightgbm_prototype,1827.139369,40692.569673,67,30,9,0,44,0


In [11]:
actual_storage = _reconstruct_storage(
    pl.from_pandas(usage_frame[["Date", "Delivery", "Total Usage"]])
).to_pandas()
actual_storage["Date"] = pd.to_datetime(actual_storage["Date"])

window_start = prototype.delivery_plan["Date"].min()
window_end = prototype.delivery_plan["Date"].max()
actual_window = actual_storage.loc[actual_storage["Date"].between(window_start, window_end)].copy()

baseline_storage = _reconstruct_storage(
    pl.from_pandas(
        baseline_delivery_plan[["Date", "optimized_delivery", "actual_total_usage"]].rename(
            columns={
                "optimized_delivery": "Delivery",
                "actual_total_usage": "Total Usage",
            }
)
    )
).to_pandas()
baseline_storage["Date"] = pd.to_datetime(baseline_storage["Date"])

actual_strategy = actual_window.loc[:, [
    "Date",
    "Delivery",
    "Total Usage",
    "ending_balance",
    "max_injection",
    "max_withdrawal",
    "min_inventory",
    "max_inventory",
    "is_month_end",
    "injection_limit_exceeded",
    "withdrawal_limit_exceeded",
    "below_min_inventory",
    "above_max_inventory",
]].copy()
baseline_strategy = baseline_storage.loc[:, [
    "Date",
    "Delivery",
    "Total Usage",
    "ending_balance",
    "max_injection",
    "max_withdrawal",
    "min_inventory",
    "max_inventory",
    "is_month_end",
    "injection_limit_exceeded",
    "withdrawal_limit_exceeded",
    "below_min_inventory",
    "above_max_inventory",
]].copy()
prototype_strategy = prototype.delivery_plan.loc[:, [
    "Date",
    "optimized_delivery",
    "actual_total_usage",
    "ending_balance",
    "max_injection",
    "max_withdrawal",
    "min_inventory",
    "max_inventory",
    "is_month_end",
    "injection_limit_exceeded",
    "withdrawal_limit_exceeded",
    "below_min_inventory",
    "above_max_inventory",
]].rename(columns={"optimized_delivery": "Delivery", "actual_total_usage": "Total Usage"})

_, _, baseline_violation_totals, baseline_violation_by_date, baseline_month_end_balance = _build_penalty_flag_report(
    actual_strategy=actual_strategy,
    planned_strategy=baseline_strategy,
    actual_label="Actual delivery",
    planned_label="Existing optimized delivery",
)

comparison_violation_totals = pd.concat(
    [
        baseline_violation_totals.loc[baseline_violation_totals["strategy"] == "Actual delivery"],
        baseline_violation_totals.loc[baseline_violation_totals["strategy"] == "Existing optimized delivery"],
        prototype.violation_totals.loc[prototype.violation_totals["strategy"] == "Rolling LightGBM prototype"],
    ],
    ignore_index=True,
)

comparison_violation_table = (
    comparison_violation_totals.pivot(index="strategy", columns="violation_type", values="violation_count")
    .fillna(0)
    .astype(int)
)
comparison_violation_table

violation_type,Any penalty day,Daily activity violation,Month-end inventory violation
strategy,,,
Actual delivery,137,136,2
Existing optimized delivery,102,93,10
Rolling LightGBM prototype,104,97,9


In [12]:
strategy_summary = pd.DataFrame(
    [
        {
            "strategy": "Actual delivery",
            "Any penalty day": int(comparison_violation_table.loc["Actual delivery", "Any penalty day"]),
            "Daily activity violation": int(comparison_violation_table.loc["Actual delivery", "Daily activity violation"]),
            "Month-end inventory violation": int(comparison_violation_table.loc["Actual delivery", "Month-end inventory violation"]),
            "balance_below_zero": int((actual_strategy["ending_balance"] < 0).sum()),
            "below_min_inventory": int(actual_strategy["below_min_inventory"].sum()),
            "above_max_inventory": int(actual_strategy["above_max_inventory"].sum()),
            "injection_limit_exceeded": int(actual_strategy["injection_limit_exceeded"].sum()),
            "withdrawal_limit_exceeded": int(actual_strategy["withdrawal_limit_exceeded"].sum()),
            "mean_ending_balance": float(actual_strategy["ending_balance"].mean()),
        },
        {
            "strategy": "Existing optimized delivery",
            "Any penalty day": int(comparison_violation_table.loc["Existing optimized delivery", "Any penalty day"]),
            "Daily activity violation": int(comparison_violation_table.loc["Existing optimized delivery", "Daily activity violation"]),
            "Month-end inventory violation": int(comparison_violation_table.loc["Existing optimized delivery", "Month-end inventory violation"]),
            "balance_below_zero": int((baseline_strategy["ending_balance"] < 0).sum()),
            "below_min_inventory": int(baseline_strategy["below_min_inventory"].sum()),
            "above_max_inventory": int(baseline_strategy["above_max_inventory"].sum()),
            "injection_limit_exceeded": int(baseline_strategy["injection_limit_exceeded"].sum()),
            "withdrawal_limit_exceeded": int(baseline_strategy["withdrawal_limit_exceeded"].sum()),
            "mean_ending_balance": float(baseline_strategy["ending_balance"].mean()),
        },
        {
            "strategy": "Rolling LightGBM prototype",
            "Any penalty day": int(comparison_violation_table.loc["Rolling LightGBM prototype", "Any penalty day"]),
            "Daily activity violation": int(comparison_violation_table.loc["Rolling LightGBM prototype", "Daily activity violation"]),
            "Month-end inventory violation": int(comparison_violation_table.loc["Rolling LightGBM prototype", "Month-end inventory violation"]),
            "balance_below_zero": int((prototype_strategy["ending_balance"] < 0).sum()),
            "below_min_inventory": int(prototype_strategy["below_min_inventory"].sum()),
            "above_max_inventory": int(prototype_strategy["above_max_inventory"].sum()),
            "injection_limit_exceeded": int(prototype_strategy["injection_limit_exceeded"].sum()),
            "withdrawal_limit_exceeded": int(prototype_strategy["withdrawal_limit_exceeded"].sum()),
            "mean_ending_balance": float(prototype_strategy["ending_balance"].mean()),
        },
    ]
)
strategy_summary.round(2)

,strategy,Any penalty day,Daily activity violation,Month-end inventory violation,balance_below_zero,below_min_inventory,above_max_inventory,injection_limit_exceeded,withdrawal_limit_exceeded,mean_ending_balance
0,Actual delivery,137,136,2,0,0,2,88,48,61853.79
1,Existing optimized delivery,102,93,10,72,10,0,55,38,40465.88
2,Rolling LightGBM prototype,104,97,9,44,9,0,67,30,40692.57


In [13]:
violation_count_compare = comparison_violation_totals.loc[
    comparison_violation_totals["violation_type"] != "Any penalty day"
].copy()

violation_chart = (
    alt.Chart(violation_count_compare)
    .mark_bar(size=34)
    .encode(
        x=alt.X("strategy:N", title=None),
        y=alt.Y("violation_count:Q", title="Violation count"),
        color=alt.Color("violation_type:N", title="Violation type"),
        xOffset="violation_type:N",
        tooltip=["strategy:N", "violation_type:N", "violation_count:Q"],
    )
    .properties(width=460, height=280, title="Violation comparison across planning policies")
)

risk_chart = (
    alt.Chart(strategy_summary)
    .mark_bar(size=48)
    .encode(
        x=alt.X("strategy:N", title=None),
        y=alt.Y("balance_below_zero:Q", title="Days with storage bank below zero"),
        color=alt.Color("strategy:N", legend=None),
        tooltip=[
            "strategy:N",
            "balance_below_zero:Q",
            alt.Tooltip("mean_ending_balance:Q", format=".1f"),
            "Any penalty day:Q",
        ],
    )
    .properties(width=320, height=280, title="Storage depletion risk")
)

violation_chart | risk_chart

alt.HConcatChart(...)

In [14]:
balance_compare = pd.concat(
    [
        actual_window.loc[:, ["Date", "ending_balance"]].assign(strategy="Actual delivery"),
        baseline_storage.loc[:, ["Date", "ending_balance"]].assign(strategy="Existing optimized delivery"),
        prototype.delivery_plan.loc[:, ["Date", "ending_balance"]].assign(strategy="Rolling LightGBM prototype"),
    ],
    ignore_index=True,
)

zero_line = pd.DataFrame({"Date": [window_start, window_end], "threshold": [0.0, 0.0]})
balance_line = alt.Chart(balance_compare).mark_line().encode(
    x=alt.X("Date:T", title=None),
    y=alt.Y("ending_balance:Q", title="Ending balance"),
    color=alt.Color("strategy:N", title="Strategy"),
    tooltip=["Date:T", "strategy:N", alt.Tooltip("ending_balance:Q", format=".1f")],
)
zero_reference = alt.Chart(zero_line).mark_line(color="#333333", strokeDash=[5, 5]).encode(
    x=alt.X("Date:T", title=None),
    y=alt.Y("threshold:Q", title="Ending balance"),
)

month_end_compare = pd.concat(
    [
        baseline_month_end_balance.loc[baseline_month_end_balance["strategy"] == "Actual delivery"],
        baseline_month_end_balance.loc[baseline_month_end_balance["strategy"] == "Existing optimized delivery"],
        prototype.month_end_balance.loc[prototype.month_end_balance["strategy"] == "Rolling LightGBM prototype"],
    ],
    ignore_index=True,
)
band_frame = month_end_compare.loc[:, ["Date", "min_inventory", "max_inventory"]].drop_duplicates()
band_lower = alt.Chart(band_frame).mark_line(strokeDash=[4, 4], color="#7f7f7f").encode(x="Date:T", y="min_inventory:Q")
band_upper = alt.Chart(band_frame).mark_line(strokeDash=[4, 4], color="#7f7f7f").encode(x="Date:T", y="max_inventory:Q")
month_end_lines = alt.Chart(month_end_compare).mark_line(point=True).encode(
    x=alt.X("Date:T", title=None),
    y=alt.Y("ending_balance:Q", title="Month-end balance"),
    color=alt.Color("strategy:N", title="Strategy"),
    tooltip=["Date:T", "strategy:N", alt.Tooltip("ending_balance:Q", format=".1f")],
)

(zero_reference + balance_line).properties(width=920, height=300, title="Daily storage balance path") & (band_lower + band_upper + month_end_lines).properties(width=920, height=260, title="Month-end inventory compliance")

alt.VConcatChart(...)

In [15]:
actual_penalty_days = int(comparison_violation_table.loc["Actual delivery", "Any penalty day"])
existing_penalty_days = int(comparison_violation_table.loc["Existing optimized delivery", "Any penalty day"])
prototype_penalty_days = int(comparison_violation_table.loc["Rolling LightGBM prototype", "Any penalty day"])
existing_negative_days = int(strategy_summary.loc[strategy_summary["strategy"] == "Existing optimized delivery", "balance_below_zero"].iloc[0])
prototype_negative_days = int(strategy_summary.loc[strategy_summary["strategy"] == "Rolling LightGBM prototype", "balance_below_zero"].iloc[0])
prototype_reduction = existing_negative_days - prototype_negative_days
penalty_day_change = existing_penalty_days - prototype_penalty_days

takeaways = f"""
## Prototype Takeaways

- Under the corrected boolean flag logic, the existing optimized plan records **{existing_penalty_days}** dates with at least one tariff violation.
- The rolling LightGBM prototype records **{prototype_penalty_days}** dates with at least one tariff violation, changing the violation-day count by **{penalty_day_change}** days relative to the existing optimized plan.
- Storage depletion remains the key operational risk. The existing optimized plan records **{existing_negative_days}** depletion days, while the rolling LightGBM prototype records **{prototype_negative_days}**, a change of **{prototype_reduction}** days.
- Actual delivery provides the benchmark at **{actual_penalty_days}** dates with at least one tariff violation in the same window.

## What Changed in the Logic

1. The comparison now focuses on whether a tariff violation happened, not on a synthetic cash-out amount.
2. The model is refit weekly instead of staying fixed for the whole year.
3. The optimizer only trusts a short modeled horizon and uses a conservative fallback demand assumption for the rest of the month.
4. A reserve floor and safety-stock uplift make it more expensive to run the storage bank close to empty.

## Recommended Next Steps

1. Validate the prototype over multiple retrain cadences such as daily, every 3 days, and weekly.
2. Tune the reserve floor and conservative uplift against violation-day and depletion-day counts, not only MAE.
3. Inspect the remaining prototype violation dates directly to see whether they are concentrated in one season or one tariff rule.
4. If depletion risk remains high, increase the reserve floor or shorten the trusted forecast horizon further.
"""
display(Markdown(takeaways))


## Prototype Takeaways

- Under the corrected boolean flag logic, the existing optimized plan records **102** dates with at least one tariff violation.
- The rolling LightGBM prototype records **104** dates with at least one tariff violation, changing the violation-day count by **-2** days relative to the existing optimized plan.
- Storage depletion remains the key operational risk. The existing optimized plan records **72** depletion days, while the rolling LightGBM prototype records **44**, a change of **28** days.
- Actual delivery provides the benchmark at **137** dates with at least one tariff violation in the same window.

## What Changed in the Logic

1. The comparison now focuses on whether a tariff violation happened, not on a synthetic cash-out amount.
2. The model is refit weekly instead of staying fixed for the whole year.
3. The optimizer only trusts a short modeled horizon and uses a conservative fallback demand assumption for the rest of the month.
4. A reserve floor and safety-stock uplift make it more expensive to run the storage bank close to empty.

## Recommended Next Steps

1. Validate the prototype over multiple retrain cadences such as daily, every 3 days, and weekly.
2. Tune the reserve floor and conservative uplift against violation-day and depletion-day counts, not only MAE.
3. Inspect the remaining prototype violation dates directly to see whether they are concentrated in one season or one tariff rule.
4. If depletion risk remains high, increase the reserve floor or shorten the trusted forecast horizon further.
